# Lab 8: Behavior Cloning with Flow Matching

In this lab we will teach a drone to fly through a field of obstacles by **imitating an expert**. The expert is the RRT planner you wrote in Lab 2. RRT needs to know exactly where every obstacle is; our drone will not. Instead, the drone has a **range finder** that measures (noisy) distances to nearby obstacles, and it must decide where to go from those measurements alone.

We will learn a policy $\pi_{\bar\theta}: \bar o \mapsto \bar a$ from a dataset of expert demonstrations $\mathcal{D} = \{(\bar o^{(i)}, \bar a^{(i)})\}_{i=1}^N$ in two ways:
1. **Regression** with a mean-squared-error loss (the baseline from last lecture), and
2. **Flow matching**, which learns the full (possibly multimodal) distribution of expert actions given the observation.

When there is an obstacle right in front of the drone, the expert sometimes goes left and sometimes goes right. As we discussed in class, a regression loss averages these two choices, and the average of "left" and "right" is "straight into the obstacle." Flow matching does not have this problem. We will compare the two methods in this assignment.

Everything is a planar, top-view simulation; there is no hardware component this week, and it is not a team assignment.

### Notation (same as Lecture 17)
| symbol | meaning | in code |
|---|---|---|
| $\bar o$ | observation (range finder + goal direction) | `obs` |
| $\bar a$ | action (a short sequence of waypoints, see Part I) | `actions` |
| $\bar a_0 \sim \mathcal N(\bar 0, I)$ | noise sample, same dimension as $\bar a$ | `a0` |
| $t \in [0, 1]$ | **flow time** (not physical time!) | `t` |
| $\bar a_t = (1-t)\,\bar a_0 + t\,\bar a^{(i)}$ | point on the line from noise to data | `a_t` |
| $\bar U_t^{\bar\theta}(\bar a, t, \bar o)$ | neural network velocity field | `model(a, t, obs)` |
| $K$ | number of Euler steps at inference | `K` |

We will use $k$ for the physical time step of the drone, to keep it separate from the flow time $t$.

### Setup
We recommend running this lab on **Google Colab** with a GPU runtime (Runtime $\to$ Change runtime type $\to$ T4 GPU). PyTorch is preinstalled there, and the two training runs in Part IV take a few minutes each. A CPU also works, but those training runs then take roughly 10 minutes each. If you prefer your `mae345` conda environment, activate it and run `pip install torch` once.

Download `rrt_demos.zip` (the expert demonstrations) from our GitHub repo and upload it to Colab with the "Files" tab on the left (or put it next to this notebook if running locally). Do not unzip it; the notebook reads it directly.

In [ ]:
import io
import time
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

%matplotlib inline
torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
# Do not modify
DATA_ZIP = "rrt_demos.zip"
with zipfile.ZipFile(DATA_ZIP) as z:
    data = dict(np.load(io.BytesIO(z.read("rrt_demos.npz"))))
for key, val in data.items():
    print(f"{key:>12}: shape {val.shape}")

# Part I: The Drone, Its Sensor, and the Expert (5 pts)

### The world
The world is the same as the synthetic example in Lab 2: a $3\,\text{m} \times 4\,\text{m}$ rectangle with $x \in [0, 3]$ and $y \in [0, 4]$, filled with 5 to 10 circular obstacles. Each obstacle is stored as a row `[cx, cy, r]` (center and radius). The drone is a disk of radius 0.1 m. It starts near the bottom edge (black $\times$) and must reach a goal circle of radius 0.25 m near the top edge (green). It crashes if it touches an obstacle or leaves the arena.

### The range finder
The drone has a range finder with 24 beams spaced evenly over $360^\circ$ (every $15^\circ$, starting along $+x$ and going counterclockwise). Each beam returns the distance to the first obstacle or wall it hits, up to a maximum range of 2 m. The measurements are imperfect:
- every reading has Gaussian noise with a standard deviation of 5 cm, and
- each beam independently "drops out" with probability 5%, in which case it returns the maximum range (as if nothing were there).

### The expert
The demonstrations were made with the RRT algorithm from Lab 2, run in the **configuration space**: every obstacle was inflated by the drone radius plus a safety margin. RRT paths are jagged, so each path was then **shortcut** (repeatedly replace a stretch of the path with a straight segment whenever that segment is collision-free). Several RRT runs were made for each obstacle field. Because RRT is randomized, these runs sometimes go around an obstacle on different sides.

The drone then flew each path while being pushed around by a random (correlated) wind. At every time step $k$, the expert's action is an **action chunk**: the next $H = 8$ waypoints along its path (spaced 0.1 m apart), written relative to the drone's current position $\bar p_k$:
$$\bar a_k = \big(\bar w_1 - \bar p_k,\; \bar w_2 - \bar p_k,\; \dots,\; \bar w_8 - \bar p_k\big) \in \mathbb{R}^{16}.$$
The drone flies to the first waypoint, the wind pushes it a bit, and the expert computes a new chunk from wherever the drone ended up. Because of the wind, the drone is often slightly off the path, so the demonstrations also show how to **recover** from small errors. Predicting a chunk of future waypoints instead of a single step is common in modern robot learning (e.g., the $\pi_0$ model from lecture predicts actions $\bar a_t, \dots, \bar a_{t+H}$); it makes the policy commit to a plan for a short while instead of changing its mind every step.

### The dataset
| key | shape | meaning |
|---|---|---|
| `obstacles` | $(E, 10, 3)$ | obstacles of each environment, padded with zero rows; use `n_obstacles` |
| `n_obstacles`, `starts`, `goals` | $(E,)$, $(E, 2)$, $(E, 2)$ | per environment |
| `demo_env`, `demo_start`, `demo_len` | $(D,)$ | environment index, first time step, and length of each demo |
| `positions` | $(T, 2)$ | drone position $\bar p_k$ at every time step (all demos concatenated) |
| `ranges` | $(T, 24)$ | noisy range finder reading recorded at every time step |
| `actions` | $(T, 8, 2)$ | expert action chunk at every time step |

The cells below contain the simulator and some plotting helpers. Read through the simulator so you know what each function does, but do not modify it.

In [ ]:
# Do not modify
import numpy as np

ARENA_W, ARENA_H = 3.0, 4.0      # arena size [m]: x in [0, 3], y in [0, 4]
DRONE_RADIUS = 0.1               # [m]
GOAL_RADIUS = 0.25               # [m]
STEP_SIZE = 0.1                  # nominal distance the drone moves per time step [m]

N_BEAMS = 24                     # range finder beams, evenly spaced over 360 degrees
MAX_RANGE = 2.0                  # [m]
BEAM_ANGLES = np.linspace(0, 2 * np.pi, N_BEAMS, endpoint=False)
BEAM_DIRS = np.stack([np.cos(BEAM_ANGLES), np.sin(BEAM_ANGLES)], axis=1)   # (N_BEAMS, 2)

RANGE_NOISE_STD = 0.05           # Gaussian noise on each range reading [m]
DROPOUT_PROB = 0.05              # probability that a beam returns no echo (reads MAX_RANGE)


def true_ranges(p, obstacles):
    """Exact distance along each beam from position p to the first obstacle or arena wall.
    p: (2,), obstacles: (M, 3) array of [cx, cy, r]. Returns (N_BEAMS,) clipped to MAX_RANGE."""
    d = BEAM_DIRS
    # walls
    with np.errstate(divide="ignore", invalid="ignore"):
        sx = np.where(d[:, 0] > 0, (ARENA_W - p[0]) / d[:, 0], np.where(d[:, 0] < 0, -p[0] / d[:, 0], np.inf))
        sy = np.where(d[:, 1] > 0, (ARENA_H - p[1]) / d[:, 1], np.where(d[:, 1] < 0, -p[1] / d[:, 1], np.inf))
    s = np.minimum(sx, sy)
    # circles: |p + s d - c|^2 = r^2  ->  s^2 + 2 b s + cc = 0
    if len(obstacles) > 0:
        pc = p[None, :] - obstacles[:, :2]                       # (M, 2)
        b = d @ pc.T                                             # (N, M)
        cc = np.sum(pc ** 2, axis=1) - obstacles[:, 2] ** 2      # (M,)
        disc = b ** 2 - cc[None, :]
        with np.errstate(invalid="ignore"):
            s_hit = -b - np.sqrt(disc)
        s_hit = np.where((disc >= 0) & (s_hit >= 0), s_hit, np.inf)
        s = np.minimum(s, s_hit.min(axis=1))
    return np.clip(s, 0.0, MAX_RANGE)


def range_finder(p, obstacles, rng):
    """Noisy range finder: Gaussian noise plus random dropouts (dropped beams read MAX_RANGE)."""
    r = true_ranges(p, obstacles) + RANGE_NOISE_STD * rng.standard_normal(N_BEAMS)
    r[rng.random(N_BEAMS) < DROPOUT_PROB] = MAX_RANGE
    return np.clip(r, 0.0, MAX_RANGE)


def in_collision(p, obstacles):
    """True if a drone of radius DRONE_RADIUS centered at p hits an obstacle or leaves the arena."""
    if (p[0] < DRONE_RADIUS or p[0] > ARENA_W - DRONE_RADIUS or
            p[1] < DRONE_RADIUS or p[1] > ARENA_H - DRONE_RADIUS):
        return True
    if len(obstacles) == 0:
        return False
    dist = np.linalg.norm(obstacles[:, :2] - p[None, :], axis=1)
    return bool(np.any(dist < obstacles[:, 2] + DRONE_RADIUS))


def random_environment(rng, n_min=5, n_max=10):
    """Random obstacle field + start + goal. Obstacles: (M, 3) array of [cx, cy, r]."""
    while True:
        n = rng.integers(n_min, n_max + 1)
        obs = []
        tries = 0
        while len(obs) < n and tries < 500:
            tries += 1
            r = rng.uniform(0.15, 0.25)
            c = np.array([rng.uniform(0.3, ARENA_W - 0.3), rng.uniform(0.9, ARENA_H - 0.9)])
            # keep obstacles from overlapping too much
            if all(np.linalg.norm(c - o[:2]) > r + o[2] + 0.1 for o in obs):
                obs.append([c[0], c[1], r])
        if len(obs) < n:
            continue
        obs = np.array(obs)
        start = np.array([rng.uniform(0.4, ARENA_W - 0.4), 0.35])
        goal = np.array([rng.uniform(0.4, ARENA_W - 0.4), ARENA_H - 0.35])
        if not in_collision(start, obs) and not in_collision(goal, obs):
            return obs, start, goal


WIND_STD = 0.02                  # wind disturbance: w_{k+1} = WIND_CORR * w_k + WIND_STD * noise
WIND_CORR = 0.8
MAX_STEP = 2 * STEP_SIZE         # the drone cannot move further than this in one time step [m]


def drone_step(p, waypoint, w, rng):
    """One time step of the drone: fly from p toward `waypoint` (at most MAX_STEP), then get pushed by
    the wind. Returns the new position and the new wind state."""
    step = waypoint - p
    n = np.linalg.norm(step)
    if n > MAX_STEP:
        step = step * MAX_STEP / n
    w = WIND_CORR * w + WIND_STD * rng.standard_normal(2)
    return p + step + w, w


H = data["actions"].shape[1]       # number of waypoints in an action chunk
ACTION_DIM = 2 * H


def get_env(e):
    # Obstacles, start and goal of environment e in the dataset.
    n = data["n_obstacles"][e]
    return data["obstacles"][e, :n], data["starts"][e], data["goals"][e]


def get_demo(i):
    # Positions, ranges and actions of demo i.
    s, l = data["demo_start"][i], data["demo_len"][i]
    return data["positions"][s:s + l], data["ranges"][s:s + l], data["actions"][s:s + l]

In [ ]:
# Do not modify
def plot_env(ax, obstacles, start=None, goal=None, title=None):
    for cx, cy, r in obstacles:
        ax.add_artist(plt.Circle((cx, cy), r, color="C0"))
    if goal is not None:
        ax.add_artist(plt.Circle(tuple(goal), GOAL_RADIUS, ec=(0.004, 0.596, 0.105), fc="none", lw=2))
    if start is not None:
        ax.plot(*start, "kx", ms=8, mew=2)
    ax.set_xlim([0, ARENA_W]); ax.set_ylim([0, ARENA_H]); ax.set_aspect("equal")
    ax.set_xticks([0, 1, 2, 3]); ax.set_yticks([0, 1, 2, 3, 4])
    if title:
        ax.set_title(title, fontsize=10)


def plot_scan(ax, p, ranges, color="r"):
    for d, r in zip(BEAM_DIRS, ranges):
        q = p + r * d
        ax.plot([p[0], q[0]], [p[1], q[1]], color=color, lw=0.7, alpha=0.6)
        ax.plot(*q, ".", color=color, ms=4)

Run the cell below to look at the demonstrations for a few environments. Each color is a different RRT run on the same obstacle field.

In [ ]:
# Do not modify
fig, axes = plt.subplots(1, 5, figsize=(17, 5))
for ax, e in zip(axes, [0, 3, 5, 9, 12]):
    plot_env(ax, *get_env(e), title=f"environment {e}")
    for i in np.where(data["demo_env"] == e)[0]:
        P, _, _ = get_demo(i)
        ax.plot(P[:, 0], P[:, 1], ".-", ms=3, lw=1)
plt.tight_layout(); plt.show()

The next cell shows what the drone "sees" at a single time step of a demonstration: the 24 range finder beams (red) and the expert's action chunk (the 8 waypoints $\bar p_k + \bar a_k$, in orange). Try a few different values of `i` and `k`. Can you spot beams that dropped out?

In [ ]:
i, k = 0, 12   # demo index and time step; feel free to change these

P, R, A = get_demo(i)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_env(axes[0], *get_env(data["demo_env"][i]), title=f"demo {i}, time step k = {k}")
axes[0].plot(P[:, 0], P[:, 1], "k-", lw=0.8, alpha=0.4)
plot_scan(axes[0], P[k], R[k])
axes[0].plot(*(P[k] + A[k]).T, "o-", color="orange", ms=4)
axes[1].bar(np.degrees(BEAM_ANGLES), R[k], width=12)
axes[1].plot(np.degrees(BEAM_ANGLES), true_ranges(P[k], get_env(data["demo_env"][i])[0]), "k_", ms=12, mew=2, label="true range")
axes[1].set(xlabel="beam angle [deg]", ylabel="range [m]", title="range finder reading"); axes[1].legend()
plt.tight_layout(); plt.show()

**Question (5 pts):** Look at the demonstrations above. (a) Find an environment where the expert demonstrations disagree about which way to go around an obstacle. Roughly where in that environment does the disagreement start? (b) The RRT expert knows the location of *every* obstacle; the drone only has its range finder and the direction to the goal. Describe a situation where the drone's observation is not enough to tell what the expert would do.

*Your answer here.*

# Part II: Building the Dataset (15 pts)

### Observations
The observation $\bar o_k \in \mathbb{R}^{26}$ is the concatenation of
- the 24 range readings divided by `MAX_RANGE` (so they lie in $[0, 1]$), and
- the vector from the drone to the goal, $(\bar g - \bar p_k)$, divided by `GOAL_SCALE` $= 2$ (so it is roughly of size 1).

Neural networks train best when their inputs are of order 1, which is why we rescale. Implement `make_observation` below. It should work both for a single time step (`ranges` of shape `(24,)`, `p` and `goal` of shape `(2,)`) and for a batch (shapes `(T, 24)`, `(T, 2)`, `(T, 2)`). Hint: concatenate along the **last** axis (`axis=-1`). Return a `float32` array.

### The dataset
Then build the arrays `O` (shape `(T, 26)`) and `A` (shape `(T, 16)`) for the whole dataset: row $j$ of `O` is the observation at time step $j$, and row $j$ of `A` is the expert action chunk at that step, flattened to $(x_1, y_1, x_2, y_2, \dots)$. You need the goal for every time step: each time step belongs to a demo (`demo_len` tells you how many steps each demo has, and the steps of the demos are stored one demo after another), and each demo belongs to an environment (`demo_env`). `np.repeat` is useful here. Do this without for loops.

In [ ]:
GOAL_SCALE = 2.0

# TODO
def make_observation(ranges, p, goal):
    '''
    ranges: (..., 24) range finder readings [m]
    p:      (..., 2) drone position
    goal:   (..., 2) goal position
    returns: (..., 26) float32 observation
    '''
    pass


# TODO: build O (T, 26) and A (T, 16)
O = None
A = None

In [ ]:
# Do not modify
T = len(data["positions"])
assert O.shape == (T, N_BEAMS + 2) and A.shape == (T, ACTION_DIM), "wrong shapes"
j = data["demo_start"][7] + 5     # time step 5 of demo 7
g = data["goals"][data["demo_env"][7]]
assert np.allclose(O[j, -2:], (g - data["positions"][j]) / GOAL_SCALE), "goal part of the observation is wrong"
assert np.allclose(O[j, :N_BEAMS], data["ranges"][j] / MAX_RANGE), "range part of the observation is wrong"
assert np.allclose(A[j, 2:4], data["actions"][j, 1]), "actions should be flattened as (x1, y1, x2, y2, ...)"
assert make_observation(data["ranges"][j], data["positions"][j], g).shape == (N_BEAMS + 2,)
print(f"Looks good! {T} (observation, action) pairs.")

### Normalizing the actions
Flow matching transports samples from $p_\text{init} = \mathcal N(\bar 0, I)$ to the data. This works best when the data is also of order 1 in every dimension, so we standardize each action dimension to zero mean and unit standrd deviation. The networks are trained on normalized actions, and we undo the normalization before sending actions to the drone. The cell below also converts everything to PyTorch tensors.

In [ ]:
# Do not modify
a_mean, a_std = A.mean(axis=0), A.std(axis=0)

def normalize_actions(a):
    return (a - a_mean) / a_std

def unnormalize_actions(a):
    return a * a_std + a_mean

obs_train = torch.tensor(O, device=device)
act_train = torch.tensor(normalize_actions(A), dtype=torch.float32, device=device)
print(obs_train.shape, act_train.shape)

# Part III: Flow Matching (35 pts)

### The velocity field network
Our network $\bar U_t^{\bar\theta}(\bar a, t, \bar o)$ takes in a (noisy) action $\bar a$, the flow time $t$, and the observation $\bar o$, and outputs a velocity with the same dimension as $\bar a$. We use a simple multilayer perceptron (MLP): the three inputs are concatenated into one vector and passed through a few fully connected layers.

The velocity field can change a lot as $t$ goes from 0 to 1, and a single input number is easy for the network to "overlook" next to the many other inputs. So instead of feeding $t$ directly, we feed a **time embedding**: the vector $\big(\sin(2^0 \pi t), \cos(2^0 \pi t), \dots, \sin(2^{5} \pi t), \cos(2^{5} \pi t)\big) \in \mathbb{R}^{12}$, which varies at several different speeds as $t$ changes. This is a standard trick in diffusion and flow-matching models.

The helpers `mlp` and `time_embedding` below are provided (do not modify). Complete the `forward` method of `VelocityField`: concatenate `a`, `time_embedding(t)`, and `obs`, and pass the result through `self.net`. The inputs are batched: `a` has shape `(B, action_dim)`, `t` has shape `(B, 1)`, `obs` has shape `(B, obs_dim)`. `torch.cat` works like `np.concatenate` (use `dim=-1`).

In [ ]:
# Do not modify
def mlp(in_dim, out_dim, hidden=512, n_hidden=3):
    layers, d = [], in_dim
    for _ in range(n_hidden):
        layers += [nn.Linear(d, hidden), nn.SiLU()]
        d = hidden
    layers.append(nn.Linear(d, out_dim))
    return nn.Sequential(*layers).to(device)


N_FREQ = 6

def time_embedding(t):
    # (B, 1) flow times -> (B, 2 * N_FREQ) features
    f = np.pi * t * 2.0 ** torch.arange(N_FREQ, device=t.device)
    return torch.cat([torch.sin(f), torch.cos(f)], dim=-1)

In [ ]:
class VelocityField(nn.Module):
    def __init__(self, action_dim, obs_dim, hidden=512):
        super().__init__()
        self.action_dim = action_dim
        self.net = mlp(action_dim + 2 * N_FREQ + obs_dim, action_dim, hidden)

    # TODO
    def forward(self, a, t, obs):
        '''
        a:   (B, action_dim) actions a_t
        t:   (B, 1) flow times
        obs: (B, obs_dim) observations
        returns: (B, action_dim) velocities U_t(a, t, obs)
        '''
        pass

### Training loss
Implement one evaluation of the flow-matching loss, following lines 3 to 6 of the training algorithm from lecture (slide 19):

$\quad$ 3: Sample noise $\bar a_0 \sim \mathcal N(\bar 0, I)$ $\qquad$ (same dimension as actions)<br>
$\quad$ 4: Sample a time $t \sim \text{Unif}([0, 1])$<br>
$\quad$ 5: $\bar a_t \leftarrow (1-t)\,\bar a_0 + t\,\bar a^{(i)}$<br>
$\quad$ 6: $L(\bar\theta) \leftarrow \big\|\bar U_t^{\bar\theta}(\bar a_t, t, \bar o^{(i)}) - (\bar a^{(i)} - \bar a_0)\big\|^2$

We do this for a whole **mini-batch** of $B$ demonstrations $(\bar o^{(i)}, \bar a^{(i)})$ at once: draw a separate $\bar a_0$ and a separate $t$ for every row of the batch, and return the **mean** of the squared error over the batch and the action dimensions (one scalar). Useful functions: `torch.randn_like`, `torch.rand`. Make sure `t` has shape `(B, 1)` so that it broadcasts against `(B, action_dim)` tensors, and create it on the same device as the data (`torch.rand(B, 1, device=actions.device)`) so that the code also runs on a GPU.

For comparison, also implement the regression (behavior cloning with MSE) loss: the mean squared error between the network's prediction `model(obs)` and the expert action.

The training loop itself (lines 1, 2, 7, 8: sampling mini-batches and taking gradient steps) is provided below in `train`.

In [ ]:
# TODO
def flow_matching_loss(model, obs, actions):
    '''
    model:   VelocityField
    obs:     (B, obs_dim) observations o^(i)
    actions: (B, action_dim) expert actions a^(i) (normalized)
    returns: scalar loss
    '''
    pass


# TODO
def regression_loss(model, obs, actions):
    '''
    model:   network mapping (B, obs_dim) observations to (B, action_dim) actions
    returns: scalar mean squared error
    '''
    pass

In [ ]:
# Do not modify
def train(model, loss_fn, obs, actions, n_steps=50000, batch_size=512, lr=1e-3, print_every=5000):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_steps)
    losses, t0 = [], time.time()
    for step in range(n_steps):                                  # 1: repeat
        idx = torch.randint(0, len(obs), (batch_size,), device=obs.device)   # 2: sample a mini-batch
        loss = loss_fn(model, obs[idx], actions[idx])             # 3-6: your loss
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()   # 7: gradient step
        losses.append(loss.item())
        if step % print_every == 0 or step == n_steps - 1:
            print(f"step {step:6d}   loss {np.mean(losses[-200:]):.4f}   ({time.time() - t0:.0f} s)")
    return losses

### Sampling (inference)
Implement the inference algorithm from lecture (slide 20), again for a whole batch of observations at once:

$\quad$ 1: Sample noise $\bar a \sim \mathcal N(\bar 0, I)$<br>
$\quad$ 2: **for** $k = 0, 1, \dots, K-1$ **do**<br>
$\quad$ 3: $\qquad t \leftarrow k / K$<br>
$\quad$ 4: $\qquad \bar a \leftarrow \bar a + \frac{1}{K}\,\bar U_t^{\bar\theta}(\bar a, t, \bar o)$ $\qquad$ (Euler step of $\frac{d\bar a_t}{dt} = \bar U_t^{\bar\theta}(\bar a_t, t, \bar o)$)<br>
$\quad$ 5: **end for**

`sample_actions` should return the final $\bar a$ (shape `(B, action_dim)`; the action dimension is stored in `model.action_dim`). As in the loss, create new tensors on `obs.device`. The decorator `@torch.no_grad()` tells PyTorch not to track gradients, since we are not training here.

In [ ]:
# TODO
@torch.no_grad()
def sample_actions(model, obs, K=10):
    '''
    model: trained VelocityField
    obs:   (B, obs_dim) observations
    K:     number of Euler steps
    returns: (B, action_dim) sampled (normalized) actions
    '''
    pass

# Part IV: Training the Drone Policies (20 pts)

Now we train the two policies on the drone demonstrations:
- `fm_model`: a `VelocityField` with `action_dim = 16` and `obs_dim = 26`, trained with `flow_matching_loss`;
- `reg_model`: an MLP from the 26-dimensional observation to the 16-dimensional action, trained with `regression_loss`.

Both networks have the same size (3 hidden layers of 512 units). Train each one with the default settings of `train` (50,000 gradient steps on mini-batches of 512). This takes a few minutes per model on a Colab GPU and roughly 10 minutes per model on a CPU. Then plot both training curves (loss vs. gradient step, log scale on the $y$-axis).

The cell after that saves the trained weights, so that you can reload them instead of retraining if your kernel restarts.

In [ ]:
torch.manual_seed(0)
# TODO: create fm_model and reg_model, and train them on (obs_train, act_train) with the default settings of `train`
fm_model = None
reg_model = None
fm_losses = None
reg_losses = None

# TODO: plot the two training curves

In [ ]:
# Do not modify. To reload instead of retraining after a restart, run the cells above Part IV (except training), then:
#   fm_model = VelocityField(ACTION_DIM, obs_train.shape[1]); fm_model.load_state_dict(torch.load("fm_model.pt", map_location=device))
#   reg_model = mlp(obs_train.shape[1], ACTION_DIM); reg_model.load_state_dict(torch.load("reg_model.pt", map_location=device))
torch.save(fm_model.state_dict(), "fm_model.pt")
torch.save(reg_model.state_dict(), "reg_model.pt")

**Question (5 pts):** The flow-matching loss ends up much higher than the regression loss. Does this mean the flow-matching policy is worse? 

*Your answer here.*

The cell below wraps both trained networks as **policies**: functions that take a single observation (a NumPy array of shape `(26,)`) and return an action chunk in meters (shape `(8, 2)`).

In [ ]:
# Do not modify
def make_fm_policy(model, K=10):
    def policy(obs):
        a = sample_actions(model, torch.as_tensor(obs, device=device)[None], K=K)[0].cpu().numpy()
        return unnormalize_actions(a).reshape(H, 2)
    return policy


def make_reg_policy(model):
    @torch.no_grad()
    def policy(obs):
        a = model(torch.as_tensor(obs, device=device)[None])[0].cpu().numpy()
        return unnormalize_actions(a).reshape(H, 2)
    return policy


fm_policy = make_fm_policy(fm_model)
reg_policy = make_reg_policy(reg_model)

# Part V: Flying the Drone (20 pts)

So far we have only trained networks. A policy is useful only if it works **in closed loop**, where every action changes the next observation. Implement `rollout`, which flies the drone with a policy from `start` until it reaches the goal, crashes, or runs out of time. The policy is used in a receding-horizon fashion:

1. Read the range finder at the current position $\bar p$ (`range_finder`) and build the observation (`make_observation`).
2. Ask the policy for an action chunk: `chunk = policy(obs)`, an array of shape `(8, 2)` of waypoints **relative to $\bar p$**.
3. Remember the position where the chunk was computed, `p_base = p`. Then fly to the first `n_exec` waypoints one after the other: for $j = 0, \dots,$ `n_exec - 1`, call `p, w = drone_step(p, p_base + chunk[j], w, rng)` (this moves the drone and applies the wind; `w` is the wind state, which starts at zero). After **every** step, append the new position to the trajectory, and stop if the drone has crashed (`in_collision`, return `"crash"`) or is within `GOAL_RADIUS` of the goal (return `"success"`).
4. Repeat from 1. If the trajectory reaches more than `max_steps` steps, return `"timeout"`.

The function returns the outcome string and the trajectory as an array of shape `(number of positions, 2)` that starts with `start`.

In [ ]:
# TODO
def rollout(policy, obstacles, start, goal, rng, n_exec=4, max_steps=200):
    '''
    policy:    function mapping an observation (26,) to an action chunk (8, 2) relative to the drone
    obstacles: (M, 3) array of [cx, cy, r]
    start, goal: (2,) arrays
    rng:       np.random.Generator (pass it to range_finder and drone_step)
    returns:   (outcome, trajectory) where outcome is "success", "crash" or "timeout"
    '''
    p, w, traj = start.copy(), np.zeros(2), [start.copy()]
    pass

In [ ]:
# Do not modify: some sanity checks with simple hand-written policies
empty = np.zeros((0, 3))
start_, goal_ = np.array([1.5, 0.35]), np.array([1.5, 3.65])
straight_up = lambda obs: np.stack([np.zeros(H), STEP_SIZE * np.arange(1, H + 1)], axis=1)   # always fly +y
hover = lambda obs: np.zeros((H, 2))

result, traj = rollout(straight_up, empty, start_, goal_, np.random.default_rng(0))
assert result == "success", f"flying straight up in an empty arena should succeed, got {result}"
assert np.allclose(traj[0], start_) and traj.shape[1] == 2, "trajectory should start at start and have shape (n, 2)"
assert np.linalg.norm(traj[-1] - goal_) < GOAL_RADIUS and np.linalg.norm(traj[-2] - goal_) >= GOAL_RADIUS, "stop as soon as the goal is reached"
assert np.allclose(traj[-1], [1.39112, 3.42698], atol=1e-4), "same random numbers should give the same flight: call range_finder once per chunk, then drone_step once per waypoint, both with rng"

result, traj = rollout(straight_up, np.array([[1.5, 1.5, 0.3]]), start_, goal_, np.random.default_rng(0))
assert result == "crash" and in_collision(traj[-1], np.array([[1.5, 1.5, 0.3]])), "should crash into the obstacle"
assert not any(in_collision(q, np.array([[1.5, 1.5, 0.3]])) for q in traj[:-1]), "stop as soon as the drone crashes"

result, traj = rollout(hover, empty, np.array([1.5, 2.0]), goal_, np.random.default_rng(0), max_steps=40)
assert result == "timeout" and 36 < len(traj) - 1 <= 44, "hovering should time out after about max_steps steps"
print("Looks good!")

The cell below flies both of your trained policies in one of the environments from the dataset. Try a few different environments and seeds.

In [ ]:
e, seed = 3, 0   # feel free to change these
fig, axes = plt.subplots(1, 2, figsize=(9, 5.5))
for ax, (name, policy) in zip(axes, [("flow matching", fm_policy), ("regression", reg_policy)]):
    torch.manual_seed(seed)
    result, traj = rollout(policy, *get_env(e), np.random.default_rng(seed))
    plot_env(ax, *get_env(e), title=f"{name}: {result}")
    ax.plot(traj[:, 0], traj[:, 1], ".-", ms=2, lw=1, color="g" if result == "success" else "r")
plt.tight_layout(); plt.show()

# Part VI: Multimodality in the Drone Policy (5 pts)

Let's put the drone in a situation like the one on slide 4 of the lecture: one obstacle straight ahead, with the goal directly behind it. The obstacle field is symmetric, so going left and going right are equally good. The drone starts close to the obstacle (its edge is 0.15 m from the obstacle, the same clearance the expert keeps), so it has to choose a side right away.

The cell below takes a single range finder reading at the drone's starting position, then
- samples 200 action chunks from the flow-matching policy (thin orange lines; each line connects the 8 waypoints of one chunk), and
- computes the regression policy's action chunk (thick red line).

In [ ]:
# Do not modify
headon_obstacles = np.array([[1.5, 1.7, 0.3], [0.4, 2.6, 0.2], [2.6, 2.6, 0.2], [0.6, 1.2, 0.15], [2.4, 1.2, 0.15]])
headon_start, headon_goal = np.array([1.5, 1.15]), np.array([1.5, 3.65])
p_query = headon_start

rng = np.random.default_rng(1)
ranges_query = range_finder(p_query, headon_obstacles, rng)
obs_query = make_observation(ranges_query, p_query, headon_goal)

obs_batch = torch.tensor(obs_query, device=device)[None].repeat(200, 1)
fm_chunks = unnormalize_actions(sample_actions(fm_model, obs_batch).cpu().numpy()).reshape(-1, H, 2)
reg_chunk = reg_policy(obs_query)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
plot_env(axes[0], headon_obstacles, headon_start, headon_goal, title="200 flow-matching samples vs. regression")
plot_scan(axes[0], p_query, ranges_query)
for c in fm_chunks:
    axes[0].plot(*(p_query + np.vstack([np.zeros(2), c])).T, color="orange", lw=0.6, alpha=0.4)
axes[0].plot(*(p_query + np.vstack([np.zeros(2), reg_chunk])).T, "r-", lw=3, label="regression")
axes[0].legend(loc="lower left")
axes[1].hist(fm_chunks[:, -1, 0], bins=40, color="orange", label="flow matching")
axes[1].axvline(reg_chunk[-1, 0], color="r", lw=3, label="regression")
axes[1].set(xlabel="sideways offset of the last waypoint [m]", ylabel="count", title="where does the chunk end up?")
axes[1].legend(); plt.tight_layout(); plt.show()

Next, let's fly each policy from the start of this environment 10 times (with different wind and sensor noise each time) using your `rollout` function.

In [ ]:
# Do not modify
fig, axes = plt.subplots(1, 2, figsize=(10, 5.5))
for ax, (name, policy) in zip(axes, [("flow matching", fm_policy), ("regression", reg_policy)]):
    outcomes = []
    plot_env(ax, headon_obstacles, headon_start, headon_goal)
    for seed in range(10):
        torch.manual_seed(seed)
        result, traj = rollout(policy, headon_obstacles, headon_start, headon_goal, np.random.default_rng(seed))
        outcomes.append(result)
        ax.plot(traj[:, 0], traj[:, 1], ".-", ms=2, lw=1, color="g" if result == "success" else "r")
    ax.set_title(f"{name}: {outcomes.count('success')}/10 successes", fontsize=10)
plt.tight_layout(); plt.show()

**Question (5 pts):** Describe the distribution of flow-matching action chunks in the first plot. How does it compare to the regression output, and how does this explain what happens in the 10 flights? Do the flow-matching flights all go around the obstacle on the same side? Why is it fine that different flights make different choices, but *not* fine for one flight to switch back and forth between "left" and "right"? How does predicting chunks of 8 waypoints and executing 4 of them before re-planning help with this?

*Your answer here.*

# Part VII: Closed-Loop Evaluation

### Success rates on new obstacle fields
The cell below flies each policy in 200 **new** random environments that were not in the training data (the same 200 for every policy), and plots a few of the flights. It takes less than a minute.

In [ ]:
# Do not modify
N_TEST = 200
test_envs = [random_environment(np.random.default_rng(10_000 + e)) for e in range(N_TEST)]

def evaluate(policy, n_exec=4, seed=0):
    torch.manual_seed(seed)
    results = [rollout(policy, *env, np.random.default_rng(seed * 1000 + e), n_exec=n_exec) for e, env in enumerate(test_envs)]
    outcomes = [r for r, _ in results]
    rates = {o: outcomes.count(o) / len(outcomes) for o in ["success", "crash", "timeout"]}
    return rates, results

t0 = time.time()
rates_fm, results_fm = evaluate(fm_policy)
rates_reg, results_reg = evaluate(reg_policy)
print(f"flow matching: {rates_fm}")
print(f"regression:    {rates_reg}")
print(f"({time.time() - t0:.0f} s)")

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
for col, e in enumerate(range(6)):
    for row, (name, results) in enumerate([("flow matching", results_fm), ("regression", results_reg)]):
        result, traj = results[e]
        plot_env(axes[row, col], *test_envs[e], title=f"{name}: {result}")
        axes[row, col].plot(traj[:, 0], traj[:, 1], ".-", ms=2, lw=1, color="g" if result == "success" else "r")
plt.tight_layout(); plt.show()

# Submission Instructions

Please submit your completed notebook (with all outputs visible) to Gradescope.